In [ ]:
from supabase import create_client
from dotenv import load_dotenv
import os, pandas as pd

import logging

logging.basicConfig()
logging.getLogger("httpx").setLevel(logging.DEBUG)

load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

default_order_columns = {
    "customers": "customer_id"
}

# Here's yet another proof why SQLAlchemy would be much more efficient and flexible tool for data loading.
# Using Supabase library only to follow the SHU/HA exercises in the weekly workbook.
def load_data_over_api(table, select="*", filterer=None, sorter=None):
    all_rows = []
    start = 0
    step = 1000 # Supabase API sets a limit to the number of rows returned
    default_order_column = default_order_columns.get(table, "id")
    
    while True:
        # Unfiltered API call.
        api_call = supabase.table(table).select(select)

        # Apply optional filters using filterer function.
        if filterer:
            api_call = filterer(api_call)

        # Add optional sorting.
        if sorter:
            api_call = sorter(api_call)
        
        # Always sort by default column after custom sortings to ensure determinism during batching.
        api_call = api_call.order(default_order_column)

        # Apply range.
        # Supabase range() is inclusive, unlike Python range. That's why -1 is needed for the second argument.
        api_call = api_call.range(start, start + step - 1)
        
        # Execute API call.
        response = api_call.execute()
        
        batch = response.data
        all_rows.extend(batch)

        # Check if last batch.
        if len(batch) < step:
            break
        
        start += step
    
    return pd.DataFrame(all_rows)

# Loading unfiltered sales data.
df_orders = load_data_over_api("sales")

# Loading unfiltered customers data.
df_customers = load_data_over_api("customers")

print(f"Orders: {len(df_orders)}, Customers: {len(df_customers)}")
df_orders.dtypes

In [ ]:
# Adding filters and sorting would make using Supabase API even more complex while trying to bypass the 1000 row limit.
df_customers_tallinn = load_data_over_api("customers", filterer=lambda api_call: api_call.eq("city", "Tallinn"))
df_orders_sorted = load_data_over_api("sales", sorter=lambda api_call: api_call.order("total_price", desc=True))
df_customers_tallinn.shape, df_orders_sorted.shape

In [ ]:
df_orders_sorted.head()

In [ ]:
df_tallinn = load_data_over_api(
    "sales",
    select="*, customers!inner(*)", # INNER JOIN customers and select all columns
    filterer=lambda api_call: api_call.eq("customers.city", "Tallinn"),
    sorter=lambda api_call: api_call.order("total_price", desc=True)
)

customers_df = pd.json_normalize(df_tallinn['customers'])

df_tallinn = pd.concat([df_tallinn.drop('customers', axis=1), customers_df], axis=1)

df_tallinn.head()

In [ ]:
df_tallinn.shape